In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    confusion_matrix,
    classification_report,
    average_precision_score,
    precision_recall_curve
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from imblearn.over_sampling import SMOTE

from xgboost import XGBClassifier
import joblib

In [ ]:
fraud = pd.read_csv("../data/processed/fraud_processed.csv")
credit = pd.read_csv("../data/processed/creditcard_processed.csv")

In [ ]:
X = fraud.drop("class", axis=1)
y = fraud["class"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
sm = SMOTE(random_state=42)

X_train_sm, y_train_sm = sm.fit_resample(
    X_train,
    y_train
)

In [ ]:
print(y_train.value_counts())
print(y_train_sm.value_counts())

In [ ]:
lr = LogisticRegression(
    max_iter=1000
)

lr.fit(X_train_sm, y_train_sm)

In [ ]:
lr_pred = lr.predict(X_test)
lr_proba = lr.predict_proba(X_test)[:,1]

In [ ]:
print("F1:", f1_score(y_test, lr_pred))

print("AUC-PR:",
      average_precision_score(y_test, lr_proba))

print(confusion_matrix(y_test, lr_pred))

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_sm, y_train_sm)

In [ ]:
rf_pred = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:,1]

In [ ]:
print("F1:", f1_score(y_test, rf_pred))

print("AUC-PR:",
      average_precision_score(y_test, rf_proba))

print(confusion_matrix(y_test, rf_pred))

In [ ]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

xgb.fit(X_train_sm, y_train_sm)

In [ ]:
xgb_pred = xgb.predict(X_test)
xgb_proba = xgb.predict_proba(X_test)[:,1]

In [ ]:
print("F1:", f1_score(y_test, xgb_pred))

print("AUC-PR:",
      average_precision_score(y_test, xgb_proba))

print(confusion_matrix(y_test, xgb_pred))

In [ ]:
results = pd.DataFrame({

    "Model": ["LogReg", "RandomForest", "XGBoost"],

    "F1": [
        f1_score(y_test, lr_pred),
        f1_score(y_test, rf_pred),
        f1_score(y_test, xgb_pred)
    ],

    "AUC_PR": [
        average_precision_score(y_test, lr_proba),
        average_precision_score(y_test, rf_proba),
        average_precision_score(y_test, xgb_proba)
    ]
})

results

In [ ]:
results.sort_values(by="AUC_PR", ascending=False)

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score

In [ ]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
scores = cross_val_score(
    xgb,
    X,
    y,
    scoring="average_precision",
    cv=skf
)

scores

In [ ]:
print("Mean AUC-PR:", scores.mean())
print("Std:", scores.std())

In [ ]:
joblib.dump(
    xgb,
    "../models/xgb_fraud_model.pkl"
)

In [ ]:
credit = pd.read_csv(
    "../data/processed/creditcard_processed.csv"
)

In [ ]:
X = credit.drop("Class", axis=1)
y = credit["Class"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
X_train_sm, y_train_sm = SMOTE(
    random_state=42
).fit_resample(X_train, y_train)

In [ ]:
xgb_credit = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    eval_metric="logloss",
    random_state=42
)

xgb_credit.fit(X_train_sm, y_train_sm)

In [ ]:
pred = xgb_credit.predict(X_test)
proba = xgb_credit.predict_proba(X_test)[:,1]

print("F1:", f1_score(y_test, pred))
print("AUC-PR:", average_precision_score(y_test, proba))

In [ ]:
joblib.dump(
    xgb_credit,
    "../models/xgb_credit_model.pkl"
)